# 🧠 Радар · мозг в Colab

**Первый запуск — самотест (сервер не нужен):**
1. Слева 📁 «Файлы» → перетащите `baltdom_radar.zip` в `/content`.
2. **Среда выполнения → Сменить тип среды → GPU** (L4 или A100; T4 тоже можно — будет модель поменьше).
3. **Среда выполнения → Выполнить всё.** Через 5–15 минут появится таблица «ждём → получили» по 8 эталонным публикациям. Скопируйте вывод в чат.

**Боевой режим:** снимите галочку `SELFTEST`, в 🔑 «Секретах» добавьте два секрета и включите им «Доступ к блокноту»:
* `RADAR_URL` — адрес моста. Его пришлёт бот сообщением «🔗 Адрес моста для Colab», он же есть в `/brain` и на вкладке «Мозг».
* `RADAR_TOKEN` — значение строки `BRIDGE_TOKEN=` из файла `.env` на ноутбуке.

Затем «Выполнить всё». Вкладку не закрывать.

**💰 Бюджет.** Colab Pro ≈ 100 CU в месяц: A100 ≈ 7,5 CU/ч (≈ 13 ч в месяц), L4 ≈ 1,7 CU/ч (≈ 58 ч). Круглосуточно Colab не потянет, поэтому здесь он «усилитель»: включаем на разбор инцидента или утренний прогон. `MAX_HOURS` выключит сессию сама, `AUTO_OFF` отдаст GPU обратно — забытая на ночь A100 съест месячный бюджет.

In [ ]:
#@title 1. Настройки { display-mode: "form" }
SELFTEST = True  #@param {type:"boolean"}
ZIP_PATH = "/content/baltdom_radar.zip"  #@param {type:"string"}
MAX_HOURS = 2  #@param {type:"number"}
AUTO_OFF = True  #@param {type:"boolean"}
FORCE_PRESET = ""  #@param ["", "Qwen3.8-27B FP8 · vLLM", "Qwen3.8-27B Q4_K_M · Ollama", "Gemma 4 12B QAT · Ollama"] {allow-input: true}
RADAR_URL = RADAR_TOKEN = None
if not SELFTEST:
    from google.colab import userdata
    RADAR_URL, RADAR_TOKEN = userdata.get("RADAR_URL"), userdata.get("RADAR_TOKEN")
    assert RADAR_URL and RADAR_TOKEN, "Добавьте секреты RADAR_URL и RADAR_TOKEN (значок 🔑 слева)"
print("Режим:", "САМОТЕСТ (без сервера)" if SELFTEST else f"боевой, сервер {RADAR_URL}")

In [ ]:
#@title 2. Видеокарта и зависимости
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "GPU нет — будет CPU-режим (медленно)"
!pip -q install selectolax feedparser httpx rapidfuzz pymorphy3 pymorphy3-dicts-ru >/dev/null

In [ ]:
#@title 3. Загрузка кода мозга
import types, sys, os, zipfile
if SELFTEST:
    assert os.path.exists(ZIP_PATH), f"Не найден {ZIP_PATH} — загрузите архив в панель «Файлы»"
    zipfile.ZipFile(ZIP_PATH).extractall("/content")
    CODE_DIR = "/content/baltdom_monitor"
    sys.path.insert(0, f"{CODE_DIR}/colab")
    import brain_worker as worker
else:
    import httpx
    r = httpx.get(f"{RADAR_URL.rstrip('/')}/api/brain/code", headers={"Authorization": f"Bearer {RADAR_TOKEN}"}, timeout=60)
    HINTS = {401: "RADAR_TOKEN не совпадает с BRIDGE_TOKEN в .env на ноутбуке",
             403: "адрес ведёт не на мост Радара — проверьте RADAR_URL",
             502: "туннель жив, но Радар на ноутбуке не запущен (start_radar.bat)",
             530: "туннель на ноутбуке не соединён с Cloudflare: адрес устарел или сеть офиса режет туннель. "
                  "Актуальный адрес и проверка — в боте, команда /brain"}
    if r.status_code != 200:
        raise SystemExit(f"❌ Мост ответил {r.status_code}: {HINTS.get(r.status_code, r.text[:200])}")
    worker = types.ModuleType("brain_worker")
    exec(compile(r.json()["worker"], "brain_worker.py", "exec"), worker.__dict__)
    sys.modules["brain_worker"] = worker
gpu, vram = worker.gpu_info()
rate = worker.cu_rate(gpu)
print(f"✅ Код загружен. GPU: {gpu} ({vram:.0f} ГБ), ≈{rate} CU/ч → 100 CU Colab Pro ≈ {100/rate:.0f} ч работы")
print("Кандидаты моделей:", [p["name"] for p in worker.pick_presets(vram, FORCE_PRESET or None)])

In [ ]:
#@title 4. Запуск
if SELFTEST:
    await worker.selftest(CODE_DIR, FORCE_PRESET or None)
else:
    await worker.main(RADAR_URL, RADAR_TOKEN, FORCE_PRESET or None, max_hours=MAX_HOURS)
if AUTO_OFF:   # при ошибке сюда не дойдём — среда останется для разбора логов
    from google.colab import runtime
    print("🔌 Готово, отключаю среду выполнения — CU больше не списываются. Вывод выше сохранится.")
    runtime.unassign()


### Если что-то пошло не так
* **«401 bad token»** — `RADAR_TOKEN` не совпадает с `BRIDGE_TOKEN` на сервере.
* **Нет связи с сервером** — сервер должен быть доступен из интернета по HTTPS (см. README → «Сеть»).
* **vLLM не поднялся** — ноутбук сам переключится на Ollama и модель поменьше; логи в `/content/vllm.log`.
* **Кончились CU** — Colab выдаст T4/CPU или откажет. Радар продолжит работу на сервере; докупите CU или подождите.